# Retrospective Pattern Mining (SNA)

依據 `strategy/retrospective_pattern_mining.md` 實作：

- Stage-wise train subset（A/B/C）
- 模型比較：Logistic / Decision Tree / LightGBM / XGBoost（可選）
- 指標主軸：PR-AUC + threshold scan + risk tier

> 本 notebook 為回顧式關聯分析（retrospective），重點是找出與 fraud 高關聯的帳戶行為模式。


In [ ]:
# ==== 0) Imports ====
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

import matplotlib.pyplot as plt

HAS_LGB = True
HAS_XGB = True
try:
    import lightgbm as lgb
except Exception:
    HAS_LGB = False

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGB = False


In [ ]:
# ==== 1) Config ====
DATA_PATH = Path('../Transactions Data.csv')
OUT_DIR = Path('../')

# Stage setup
STAGE = 'A'  # A/B/C
STAGE_TRAIN_FRAC = {'A': 0.2, 'B': 0.5, 'C': 1.0}

USE_SAMPLE = True
SAMPLE_FRAC = 0.20
RANDOM_STATE = 42

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO  = 0.15

RISK_T_MID = 0.10
RISK_T_HIGH = 0.30


In [ ]:
# ==== 2) Load data ====
df = pd.read_csv(DATA_PATH)

if USE_SAMPLE:
    # stratified-like by step, safe implementation
    sampled = []
    for _, g in df.groupby('step', sort=False):
        if len(g) <= 1:
            sampled.append(g)
        else:
            sampled.append(g.sample(frac=min(SAMPLE_FRAC, 1.0), random_state=RANDOM_STATE))
    df = pd.concat(sampled, ignore_index=True)

df = df.sort_values('step').reset_index(drop=True)
print('Shape:', df.shape)
print('Fraud rate:', df['isFraud'].mean())


## 3) Account-level retrospective features

本段建立帳戶級行為與 SNA proxy 特徵（回顧式）：
- degree / unique counterparty
- amount stats（sum/mean/std）
- concentration / entropy
- burst / inactivity gap proxy
- pair interaction features


In [ ]:
# ==== 3) Feature engineering (retrospective account behavior + SNA proxy) ====
work = df.copy()

# prepare derived transaction fields
work['deltaOrig'] = work['oldbalanceOrg'] - work['newbalanceOrig']
work['deltaDest'] = work['newbalanceDest'] - work['oldbalanceDest']
work['amount_log1p'] = np.log1p(work['amount'].clip(lower=0))

n = len(work)
orig = work['nameOrig'].values
dest = work['nameDest'].values
amt = work['amount'].values.astype(float)
step = work['step'].values.astype(int)

# states
out_deg = defaultdict(int)
in_deg = defaultdict(int)
out_sum = defaultdict(float)
in_sum = defaultdict(float)
out_sum_sq = defaultdict(float)
in_sum_sq = defaultdict(float)
out_last_step = defaultdict(lambda: -1)
in_last_step = defaultdict(lambda: -1)
orig_to_dest = defaultdict(set)
dest_from_orig = defaultdict(set)
orig_dest_count = defaultdict(lambda: defaultdict(int))  # for concentration/entropy
pair_count = defaultdict(int)
pair_sum = defaultdict(float)

# arrays
f = {}
feat_names = [
    'orig_out_degree','dest_in_degree','orig_unique_counterparty','dest_unique_counterparty',
    'orig_out_amount_sum','dest_in_amount_sum','orig_out_amount_mean','dest_in_amount_mean',
    'orig_out_amount_std','dest_in_amount_std',
    'orig_repeat_counterparty_ratio','dest_repeat_counterparty_ratio',
    'orig_counterparty_entropy','dest_counterparty_entropy',
    'orig_recent_gap','dest_recent_gap',
    'orig_activity_burst_proxy','dest_activity_burst_proxy',
    'pair_txn_count','pair_amount_mean','is_first_time_pair'
]
for k in feat_names:
    f[k] = np.zeros(n, dtype=float)

for i in range(n):
    o = orig[i]
    d = dest[i]
    a = amt[i]
    s = max(step[i], 1)
    p = (o, d)

    # history read
    od = out_deg[o]
    dd = in_deg[d]
    ous = out_sum[o]
    dis = in_sum[d]
    ousq = out_sum_sq[o]
    disq = in_sum_sq[d]

    o_uc = len(orig_to_dest[o])
    d_uc = len(dest_from_orig[d])

    # std from sum/sum_sq/count
    o_var = max(0.0, (ousq / max(od,1)) - (ous / max(od,1))**2)
    d_var = max(0.0, (disq / max(dd,1)) - (dis / max(dd,1))**2)

    # entropy
    def entropy_from_counter(counter_dict, total):
        if total <= 0 or len(counter_dict)==0:
            return 0.0
        probs = np.array(list(counter_dict.values()), dtype=float) / total
        probs = probs[probs > 0]
        return float(-(probs * np.log(probs)).sum())

    o_ent = entropy_from_counter(orig_dest_count[o], max(od,1))
    # for dest entropy build reverse from counts in dest_from_orig not tracked by counts, approximate using in_deg + unique
    # here use repeat ratio surrogate and unique count
    d_ent = np.log(max(d_uc,1)) if d_uc > 0 else 0.0

    # gaps
    o_gap = (s - out_last_step[o]) if out_last_step[o] >= 0 else 0
    d_gap = (s - in_last_step[d]) if in_last_step[d] >= 0 else 0

    # pair
    pc = pair_count[p]
    pmean = pair_sum[p] / pc if pc > 0 else 0.0

    f['orig_out_degree'][i] = od
    f['dest_in_degree'][i] = dd
    f['orig_unique_counterparty'][i] = o_uc
    f['dest_unique_counterparty'][i] = d_uc

    f['orig_out_amount_sum'][i] = ous
    f['dest_in_amount_sum'][i] = dis
    f['orig_out_amount_mean'][i] = ous / max(od,1)
    f['dest_in_amount_mean'][i] = dis / max(dd,1)

    f['orig_out_amount_std'][i] = np.sqrt(o_var)
    f['dest_in_amount_std'][i] = np.sqrt(d_var)

    f['orig_repeat_counterparty_ratio'][i] = 1.0 - (o_uc / max(od,1))
    f['dest_repeat_counterparty_ratio'][i] = 1.0 - (d_uc / max(dd,1))

    f['orig_counterparty_entropy'][i] = o_ent
    f['dest_counterparty_entropy'][i] = d_ent

    f['orig_recent_gap'][i] = o_gap
    f['dest_recent_gap'][i] = d_gap

    f['orig_activity_burst_proxy'][i] = od / s
    f['dest_activity_burst_proxy'][i] = dd / s

    f['pair_txn_count'][i] = pc
    f['pair_amount_mean'][i] = pmean
    f['is_first_time_pair'][i] = 1.0 if pc == 0 else 0.0

    # update
    out_deg[o] += 1
    in_deg[d] += 1
    out_sum[o] += a
    in_sum[d] += a
    out_sum_sq[o] += a*a
    in_sum_sq[d] += a*a
    out_last_step[o] = s
    in_last_step[d] = s
    orig_to_dest[o].add(d)
    dest_from_orig[d].add(o)
    orig_dest_count[o][d] += 1

    pair_count[p] += 1
    pair_sum[p] += a

for k,v in f.items():
    work[k] = v

# account-level retrospective flag (for separate exploratory summaries if needed)
# here kept as tx-level target for model comparison
TARGET = 'isFraud'

print('Feature engineering done:', len(feat_names), 'retrospective features added')
work[feat_names[:8]].head()


In [ ]:
# ==== 4) Time split + stage-wise train subset ====
steps = np.sort(work['step'].unique())
train_cut = int(len(steps) * TRAIN_RATIO)
valid_cut = int(len(steps) * (TRAIN_RATIO + VALID_RATIO))

train_max_step = steps[train_cut - 1]
valid_max_step = steps[valid_cut - 1]

train_mask_full = work['step'] <= train_max_step
valid_mask = (work['step'] > train_max_step) & (work['step'] <= valid_max_step)
test_mask = work['step'] > valid_max_step

train_frac = STAGE_TRAIN_FRAC[STAGE]

X_all = work.copy()
y_all = work[TARGET].values

X_train_full = X_all.loc[train_mask_full].copy()
y_train_full = y_all[train_mask_full]
X_valid = X_all.loc[valid_mask].copy()
y_valid = y_all[valid_mask]
X_test = X_all.loc[test_mask].copy()
y_test = y_all[test_mask]

if train_frac < 1.0:
    train_df = X_train_full.copy()
    train_df[TARGET] = y_train_full
    sampled_train = (
        train_df.groupby(TARGET, group_keys=False)
                .sample(frac=train_frac, random_state=RANDOM_STATE)
                .sample(frac=1.0, random_state=RANDOM_STATE)
                .reset_index(drop=True)
    )
    X_train = sampled_train.drop(columns=[TARGET])
    y_train = sampled_train[TARGET].values
else:
    X_train = X_train_full.copy()
    y_train = y_train_full.copy()

print(f"Stage={STAGE}, train_frac={train_frac}")
print('Train:', X_train.shape, 'fraud rate:', y_train.mean())
print('Valid:', X_valid.shape, 'fraud rate:', y_valid.mean())
print('Test :', X_test.shape, 'fraud rate:', y_test.mean())


In [ ]:
# ==== 5) Feature sets ====
base_num = [
    'step','amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest',
    'isFlaggedFraud','deltaOrig','deltaDest'
]
base_cat = ['type']

retro_num = feat_names

feature_cols = base_num + base_cat + retro_num

X_train = X_train[feature_cols].copy()
X_valid = X_valid[feature_cols].copy()
X_test = X_test[feature_cols].copy()

print('Total feature columns:', len(feature_cols))


In [ ]:
# ==== 6) Evaluation helpers ====
def evaluate_probs(y_true, y_prob, model_name='model'):
    ap = average_precision_score(y_true, y_prob)
    roc = roc_auc_score(y_true, y_prob)
    return {'model': model_name, 'pr_auc': ap, 'roc_auc': roc}

def threshold_scan(y_true, y_prob, model_name='model', thresholds=None):
    if thresholds is None:
        thresholds = np.array([0.01,0.02,0.05,0.1,0.15,0.2,0.3,0.5])
    rows=[]
    y_true = np.asarray(y_true)
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp = ((y_true==1)&(pred==1)).sum()
        fp = ((y_true==0)&(pred==1)).sum()
        fn = ((y_true==1)&(pred==0)).sum()
        precision = tp / max(tp+fp,1)
        recall = tp / max(tp+fn,1)
        f1 = 2*precision*recall / max(precision+recall,1e-12)
        rows.append({'model':model_name,'threshold':t,'precision':precision,'recall':recall,'f1':f1,'pred_pos':int((pred==1).sum())})
    return pd.DataFrame(rows)

def risk_tier_summary(y_true, y_prob, model_name='model', t_mid=0.10, t_high=0.30):
    df = pd.DataFrame({'y':y_true, 'p':y_prob})
    df['tier'] = pd.cut(df['p'], bins=[-np.inf,t_mid,t_high,np.inf], labels=['Low','Medium','High'])
    out = df.groupby('tier', observed=True).agg(volume=('y','size'), fraud_count=('y','sum'), fraud_rate=('y','mean')).reset_index()
    out['model'] = model_name
    return out[['model','tier','volume','fraud_count','fraud_rate']]


def plot_pr(y_true, y_prob, title='PR Curve'):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.figure(figsize=(6,4))
    plt.plot(r,p,label=f'AP={ap:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.grid(alpha=0.2)
    plt.legend()
    plt.show()


In [ ]:
# ==== 7) Train models ====
metrics = []
scan_tables = []
tier_tables = []
pred_valid = {}

# preprocess for linear/tree
pre_lr = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), base_num + retro_num),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]), base_cat)
])
pre_tree = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median'))]), base_num + retro_num),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]), base_cat)
])

# Logistic
logit = Pipeline([
    ('prep', pre_lr),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga', n_jobs=-1, random_state=RANDOM_STATE))
])
logit.fit(X_train, y_train)
p = logit.predict_proba(X_valid)[:,1]
pred_valid['Logistic'] = p
metrics.append(evaluate_probs(y_valid, p, 'Logistic'))
scan_tables.append(threshold_scan(y_valid, p, 'Logistic'))
tier_tables.append(risk_tier_summary(y_valid, p, 'Logistic', RISK_T_MID, RISK_T_HIGH))

# Decision Tree
dtree = Pipeline([
    ('prep', pre_tree),
    ('clf', DecisionTreeClassifier(max_depth=10, min_samples_leaf=200, class_weight='balanced', random_state=RANDOM_STATE))
])
dtree.fit(X_train, y_train)
p = dtree.predict_proba(X_valid)[:,1]
pred_valid['DecisionTree'] = p
metrics.append(evaluate_probs(y_valid, p, 'DecisionTree'))
scan_tables.append(threshold_scan(y_valid, p, 'DecisionTree'))
tier_tables.append(risk_tier_summary(y_valid, p, 'DecisionTree', RISK_T_MID, RISK_T_HIGH))

# LightGBM
if HAS_LGB:
    Xtr_lgb = X_train.copy(); Xva_lgb = X_valid.copy()
    Xtr_lgb['type'] = Xtr_lgb['type'].astype('category')
    Xva_lgb['type'] = Xva_lgb['type'].astype('category')
    neg = (y_train==0).sum(); pos = (y_train==1).sum()
    spw = max(1.0, neg / max(pos,1))

    lgbm = lgb.LGBMClassifier(
        objective='binary', n_estimators=500, learning_rate=0.05, num_leaves=64,
        min_child_samples=100, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1
    )
    lgbm.fit(Xtr_lgb, y_train, eval_set=[(Xva_lgb,y_valid)], eval_metric='aucpr',
             categorical_feature=['type'], callbacks=[lgb.early_stopping(50, verbose=False)])
    p = lgbm.predict_proba(Xva_lgb)[:,1]
    pred_valid['LightGBM'] = p
    metrics.append(evaluate_probs(y_valid, p, 'LightGBM'))
    scan_tables.append(threshold_scan(y_valid, p, 'LightGBM'))
    tier_tables.append(risk_tier_summary(y_valid, p, 'LightGBM', RISK_T_MID, RISK_T_HIGH))

# XGBoost optional
if HAS_XGB:
    xtr = pd.get_dummies(X_train.copy(), columns=['type'], drop_first=False)
    xva = pd.get_dummies(X_valid.copy(), columns=['type'], drop_first=False)
    xva = xva.reindex(columns=xtr.columns, fill_value=0)

    neg = (y_train==0).sum(); pos = (y_train==1).sum()
    spw = max(1.0, neg / max(pos,1))

    xgb = XGBClassifier(
        objective='binary:logistic', eval_metric='aucpr', n_estimators=350, learning_rate=0.05,
        max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist'
    )
    xgb.fit(xtr, y_train, eval_set=[(xva,y_valid)], verbose=False)
    p = xgb.predict_proba(xva)[:,1]
    pred_valid['XGBoost'] = p
    metrics.append(evaluate_probs(y_valid, p, 'XGBoost'))
    scan_tables.append(threshold_scan(y_valid, p, 'XGBoost'))
    tier_tables.append(risk_tier_summary(y_valid, p, 'XGBoost', RISK_T_MID, RISK_T_HIGH))

metrics_df = pd.DataFrame(metrics).sort_values('pr_auc', ascending=False)
metrics_df


In [ ]:
# ==== 8) PR curves ====
for m, p in pred_valid.items():
    plot_pr(y_valid, p, title=f'{m} - Valid PR Curve')


In [ ]:
# ==== 9) Threshold scan & Risk tier outputs ====
threshold_scan_df = pd.concat(scan_tables, ignore_index=True)
risk_tier_df = pd.concat(tier_tables, ignore_index=True)

display(threshold_scan_df.head(20))
display(risk_tier_df)


In [ ]:
# ==== 10) Export artifacts ====
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUT_DIR / 'account_pattern_model_importance.csv'  # name aligned with strategy; store model metrics here
scan_path = OUT_DIR / 'account_pattern_threshold_scan.csv'
tier_path = OUT_DIR / 'account_pattern_risk_tier_summary.csv'

metrics_df.to_csv(metrics_path, index=False)
threshold_scan_df.to_csv(scan_path, index=False)
risk_tier_df.to_csv(tier_path, index=False)

# account-level feature table (retrospective)
account_feature_cols = [
    'nameOrig','nameDest','isFraud','step','amount','type'
] + retro_num
work[account_feature_cols].to_csv(OUT_DIR / 'account_feature_table.csv', index=False)

print('Saved:')
print(metrics_path)
print(scan_path)
print(tier_path)
print(OUT_DIR / 'account_feature_table.csv')


## Notes

- 這份 notebook 是回顧式關聯分析，重點是找行為模式與風險特徵，不是即時上線版因果預測。  
- 若要上線，需再做 strictly past-only、窗口化、walk-forward 版本。
